# # NFL Game Predictor
 
 This notebook predicts NFL game outcomes using ELO ratings and machine learning.
 It handles both past games (with stats) and future games (without stats).

## 1. Imports and Data Loading

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score

nfl_data = pd.read_csv('nfl_schedule_stats_2020_2025.csv', index_col=0)
nfl_data['Date'] = pd.to_datetime(nfl_data['Date'])
nfl_data.columns

Index(['Gtm', 'Week', 'Date', 'Day', 'HorA', 'Opp', 'Rslt', 'Pts', 'PtsO',
       'OT', 'Cmp', 'Att', 'Cmp%', 'Yds', 'TD', 'Y/A', 'AY/A', 'Rate', 'Sk',
       'Yds.1', 'Att.1', 'Yds.2', 'TD.1', 'Y/A.1', 'Ply', 'Tot', 'Y/P', 'FGA',
       'FGM', 'XPA', 'XPM', 'Pnt', 'Yds.3', 'Pass', 'Rsh', 'Pen', '1stD',
       '3DConv', '3DAtt', '4DConv', '4DAtt', 'Pen.1', 'Yds.4', 'FL', 'Int',
       'TO', 'ToP', 'Team', 'Year'],
      dtype='object')

In [ ]:
nfl_data.sort_values(by=['Team','Date'])

,Gtm,Week,Date,Day,HorA,Opp,Rslt,Pts,PtsO,OT,...,4DConv,4DAtt,Pen.1,Yds.4,FL,Int,TO,ToP,Team,Year
Rk,,,,,,,,,,,,,,,,,,,,,
1.0,1.0,1.0,2020-09-13,Sun,NaN,SEA,L,25,38.0,NaN,...,0.0,4.0,6.0,72.0,1.0,1.0,2.0,29:24,atl,2020
2.0,2.0,2.0,2020-09-20,Sun,@,DAL,L,39,40.0,NaN,...,2.0,2.0,8.0,51.0,0.0,0.0,0.0,33:48,atl,2020
3.0,3.0,3.0,2020-09-27,Sun,NaN,CHI,L,26,30.0,NaN,...,0.0,0.0,7.0,75.0,0.0,1.0,1.0,25:10,atl,2020
4.0,4.0,4.0,2020-10-05,Mon,@,GNB,L,16,30.0,NaN,...,2.0,4.0,3.0,23.0,0.0,0.0,0.0,30:22,atl,2020
5.0,5.0,5.0,2020-10-11,Sun,NaN,CAR,L,16,23.0,NaN,...,1.0,1.0,3.0,40.0,0.0,1.0,1.0,26:55,atl,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14.0,14.0,15.0,2025-12-14,Sun,@,NYG,NaN,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,was,2025
15.0,15.0,16.0,2025-12-20,Sat,NaN,PHI,NaN,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,was,2025
16.0,16.0,17.0,2025-12-25,Thu,NaN,DAL,NaN,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,was,2025


 ## 2. Data Cleaning and Feature Engineering

In [ ]:
replace_dict = {
    'ari': 'crd', 'lar': 'ram', 'lvr': 'rai', 'bal': 'rav',
    'hou': 'htx', 'ten': 'oti', 'lac': 'sdg', 'ind': 'clt'
}
nfl_data = nfl_data.dropna(subset=['Date'])
nfl_data['Opp'] = nfl_data['Opp'].replace(replace_dict)
nfl_data['PD'] = nfl_data['Pts'] - nfl_data['PtsO']
nfl_data = nfl_data.dropna(subset=["Week"])
nfl_data["Opp_code"] = nfl_data["Opp"].astype("category").cat.codes
nfl_data["Day_code"] = nfl_data["Day"].astype("category").cat.codes
nfl_data["target"] = (nfl_data["Rslt"] == "W").astype(int)
nfl_data["HorA_code"] = (nfl_data["HorA"] == "@").astype(int)
nfl_data['win_streak'] = (
    nfl_data.groupby('Team')['target']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

## 3. ELO Calculation Functions

In [4]:
BASE_RATING = 1500
K = 20
HOME_FIELD_ADVANTAGE = 50

def calculate_expected_result(rating_a, rating_b, home_field_advantage):
    return 1 / (1 + 10 ** ((rating_b - rating_a + home_field_advantage) / 400))

def calculate_mov_multiplier(point_diff, elo_diff):
    return np.log(abs(point_diff) + 1) * (2.2 / (elo_diff * 0.001 + 2.2))

## 4. ELO Ratings Calculation

In [5]:
elo_ratings = {}
history = []

for index, row in nfl_data.iterrows():
    team = row['Team']
    opp = row['Opp']
    week = row['Week']
    date = row['Date']
    point_diff = row['PD']
    HorA = row['HorA_code']

    elo_team = elo_ratings.get(team, BASE_RATING)
    elo_opp = elo_ratings.get(opp, BASE_RATING)

    home_field_advantage = HOME_FIELD_ADVANTAGE if HorA == 1 else -HOME_FIELD_ADVANTAGE

    if pd.isna(point_diff):
        history.append({
            'Year': row['Year'],
            'Week': week,
            'Date': date,
            "Team": team,
            "Opp": opp,
            "Team_Elo_Before": elo_team,
            "Team_Elo_After": elo_team,
            "Opp_Elo_Before": elo_opp,
            "Opp_Elo_After": elo_opp,
        })
    else:
        expected_result = calculate_expected_result(elo_team, elo_opp, home_field_advantage)
        mov_multiplier = calculate_mov_multiplier(point_diff, elo_team - elo_opp)
        score = 1 if point_diff > 0 else 0 if point_diff < 0 else 0.5
        elo_change = K * (score - expected_result) * mov_multiplier

        elo_ratings[team] = elo_team + elo_change
        elo_ratings[opp] = elo_opp - elo_change

        history.append({
            'Year': row['Year'],
            'Week': week,
            'Date': date,
            "Team": team,
            "Opp": opp,
            "Team_Elo_Before": elo_team,
            "Team_Elo_After": elo_ratings[team],
            "Opp_Elo_Before": elo_opp,
            "Opp_Elo_After": elo_ratings[opp],
        })

elo_history_data = pd.DataFrame(history)
elo_history_data.sort_values(by=['Team', 'Date'], inplace=True)

## 5. Merge ELO Ratings with Main Data

In [6]:
nfl_data = nfl_data.merge(
    elo_history_data[
        ["Year", "Week", "Team", "Team_Elo_After", "Opp_Elo_After", "Team_Elo_Before", "Opp_Elo_Before"]
    ],
    on=["Year", "Week", "Team"],
    how="left"
)
nfl_data = nfl_data.sort_values(by=["Team", "Date"])
nfl_data.head()


,Gtm,Week,Date,Day,HorA,Opp,Rslt,Pts,PtsO,OT,...,Year,PD,Opp_code,Day_code,target,HorA_code,Team_Elo_After,Opp_Elo_After,Team_Elo_Before,Opp_Elo_Before
101,1.0,1.0,2020-09-13,Sun,NaN,SEA,L,25,38.0,NaN,...,2020,-13.0,27,3,0,0,1477.015566,1633.658471,1500.000000,1610.674038
102,2.0,2.0,2020-09-20,Sun,@,DAL,L,39,40.0,NaN,...,2020,-1.0,8,3,0,1,1470.180358,1430.342905,1477.015566,1423.507697
103,3.0,3.0,2020-09-27,Sun,NaN,CHI,L,26,30.0,NaN,...,2020,-4.0,5,3,0,0,1451.204119,1473.190418,1470.180358,1454.214179
104,4.0,4.0,2020-10-05,Mon,@,GNB,L,16,30.0,NaN,...,2020,-14.0,11,1,0,1,1433.360343,1552.413587,1451.204119,1534.569811
105,5.0,5.0,2020-10-11,Sun,NaN,CAR,L,16,23.0,NaN,...,2020,-7.0,4,3,0,0,1415.380201,1564.500625,1433.360343,1546.520483


## 6. Rolling Averages for Past Games

In [7]:
def rolling_avgs(team, cols, new_cols):
    team = team.sort_values(by="Date")
    rolling_stats = team[cols].rolling(3, closed="left").mean()
    team[new_cols] = rolling_stats
    team = team.dropna(subset=new_cols)
    team = team[team['Date'] <= pd.Timestamp('2025-09-01')]
    return team

cols = ["PD", "Yds", "Rate", "Y/P", "Opp_Elo_Before", "Team_Elo_Before"]
new_cols = [f"{c}_rolling_avg" for c in cols]

teams_rolling = nfl_data.groupby("Team").apply(lambda x: rolling_avgs(x, cols, new_cols)).droplevel("Team")
teams_rolling.index = range(teams_rolling.shape[0])

/var/folders/jw/fvqmmpm91ql4538gn31g47800000gn/T/ipykernel_13441/2098658167.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  teams_rolling = nfl_data.groupby("Team").apply(lambda x: rolling_avgs(x, cols, new_cols)).droplevel("Team")


## 7. Model Training and Prediction

In [ ]:
rf = RandomForestClassifier(n_estimators=100, min_samples_split=10, random_state=1)

predictors = ['HorA_code', 'Opp_code', 'Day_code', 'Team_Elo_Before', 'Opp_Elo_Before', 'Sk', 'TO'] + new_cols
future_predictors = ['HorA_code', 'Opp_code', 'Day_code', 'Team_Elo_Before', 'Opp_Elo_Before']

def make_predictions(data, predictors, future_predictors):
    train = data[data['Date'] < pd.Timestamp('2024-09-01')]
    test = data[
        (data['Date'] >= pd.Timestamp('2024-09-01')) & 
        (data['Date'] <= pd.Timestamp('2025-09-01'))
    ]
    rf.fit(train[predictors], train['target'])

    test_past = test.dropna(subset=new_cols)
    test_future = test[test[new_cols].isnull().any(axis=1)]

    preds_past = rf.predict(test_past[predictors])

    # Ensure all future_predictors columns exist and fill NaNs with 0
    for col in future_predictors:
        if col not in test_future.columns:
            test_future[col] = 0
    # Select only the columns needed for prediction, in the correct order
    test_future_predict = test_future[future_predictors].fillna(0)

    if len(test_future_predict) > 0:
        preds_future = rf.predict(test_future_predict)
        combined_future = pd.DataFrame({
            'actual': test.loc[test_future.index, "target"], 
            'prediction': preds_future,
            'Date': test.loc[test_future.index, "Date"],
            'Team': test.loc[test_future.index, "Team"],
            'Opp': test.loc[test_future.index, "Opp"],
        }, index=test_future.index)
    else:
        combined_future = pd.DataFrame(columns=['actual', 'prediction', 'Date', 'Team', 'Opp'])

    combined_past = pd.DataFrame({
        'actual': test_past["target"], 
        'prediction': preds_past,
        'Date': test_past["Date"],
        'Team': test_past["Team"],
        'Opp': test_past["Opp"],
    }, index=test_past.index)

    combined = pd.concat([combined_past, combined_future]).sort_index()
    combined = combined.dropna(subset=["actual", "prediction"])

    # Only compute precision on rows with known outcomes (past games)
    past_mask = combined["actual"].isin([0, 1])
    precision = precision_score(
        combined.loc[past_mask, "actual"].astype(int),
        combined.loc[past_mask, "prediction"].astype(int)
    )
    return combined, precision

combined, precision = make_predictions(teams_rolling, predictors, future_predictors)

/var/folders/jw/fvqmmpm91ql4538gn31g47800000gn/T/ipykernel_13441/2246036617.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([combined_past, combined_future]).sort_index()


## 8. Results Visualization

In [9]:
combined = combined.sort_values(by=["Team", "Date"])
combined = combined.merge(
    nfl_data[["Date", "Team", "Opp", "Rslt"]],
    on=["Date", "Team", "Opp"],
    how="left"
)

print("Prediction Precision:", precision)
print(combined.head(34))

pd.crosstab(combined["actual"], combined["prediction"])

Prediction Precision: 0.7111913357400722
   actual prediction       Date Team  Opp Rslt
0       0          0 2024-09-08  atl  PIT    L
1       1          1 2024-09-16  atl  PHI    W
2       0          0 2024-09-22  atl  KAN    L
3       1          0 2024-09-29  atl  NOR    W
4       1          0 2024-10-03  atl  TAM    W
5       1          1 2024-10-13  atl  CAR    W
6       0          0 2024-10-20  atl  SEA    L
7       1          1 2024-10-27  atl  TAM    W
8       1          1 2024-11-03  atl  DAL    W
9       0          0 2024-11-10  atl  NOR    L
10      0          0 2024-11-17  atl  DEN    L
11      0          0 2024-12-01  atl  LAC    L
12      0          0 2024-12-08  atl  MIN    L
13      1          0 2024-12-16  atl  LVR    W
14      1          1 2024-12-22  atl  NYG    W
15      0          0 2024-12-29  atl  WAS    L
16      0          1 2025-01-05  atl  CAR    L
17      1          1 2024-09-08  buf  ARI    W
18      1          1 2024-09-12  buf  MIA    W
19      1          

prediction,0,1
actual,,
0,192,80
1,75,197
